# Score Méta par Archetype — Phase 3

**Objectif :** calculer un score méta par archetype qui combine :
- **Fréquence** : quelle part des decks de tournoi cet archetype représente
- **Placement** : est-ce que cet archetype *gagne* les tournois ou juste y participe
- **Trend** : est-il en hausse ou en chute sur les 3 derniers mois

Ce score servira de cible (target) pour le modèle prédictif du notebook 05.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

con = sqlite3.connect('../data/yugioh.db')

# Charger tous les decks avec date et placement
decks = pd.read_sql("""
    SELECT id, archetype, placement, uploaded
    FROM tournament_decks
    WHERE illegal = 0
      AND archetype IS NOT NULL
      AND placement IS NOT NULL
      AND uploaded IS NOT NULL
""", con)

decks['date'] = pd.to_datetime(decks['uploaded'].str[:10])
decks['month'] = decks['date'].dt.to_period('M')

print(f'Decks chargés    : {len(decks):,}')
print(f'Archetypes uniques : {decks["archetype"].nunique()}')
print(f'Période          : {decks["date"].min().date()} → {decks["date"].max().date()}')

## 1. Score méta mensuel par archetype

In [ ]:
# Pour chaque mois, calculer :
#   - deck_count     : nombre de decks de cet archetype
#   - total_decks    : total decks ce mois-là (toutes archetypes)
#   - share          : deck_count / total_decks
#   - avg_placement  : placement moyen (1er = meilleur)
#   - placement_score: 1 / avg_placement (plus élevé = meilleur)
#   - meta_score     : share * placement_score (normalisé entre 0 et 1)

monthly_total = decks.groupby('month').size().rename('total_decks')

monthly = (decks.groupby(['month', 'archetype'])
           .agg(deck_count=('id', 'count'),
                avg_placement=('placement', 'mean'))
           .reset_index())

monthly = monthly.join(monthly_total, on='month')
monthly['share'] = monthly['deck_count'] / monthly['total_decks']
monthly['placement_score'] = 1.0 / monthly['avg_placement']

# Normaliser placement_score par mois (max = 1 ce mois-là)
monthly['placement_score_norm'] = (monthly
    .groupby('month')['placement_score']
    .transform(lambda x: x / x.max()))

# meta_score = moyenne géométrique de share et placement_score_norm
# → pénalise les archetypes qui jouent beaucoup mais placent mal
monthly['meta_score'] = np.sqrt(monthly['share'] * monthly['placement_score_norm'])

print(f'Lignes (mois × archetype) : {len(monthly):,}')
print(f'Mois couverts             : {monthly["month"].nunique()}')
print()

# Top archetypes sur le dernier mois disponible
last_month = monthly['month'].max()
top_last = (monthly[monthly['month'] == last_month]
            .sort_values('meta_score', ascending=False)
            .head(15))

print(f'Top 15 — {last_month} :')
print(top_last[['archetype', 'deck_count', 'share', 'avg_placement', 'meta_score']]
      .to_string(index=False, float_format='{:.3f}'.format))

## 2. Trend — archetypes en hausse ou en chute

In [ ]:
REFERENCE_DATE = pd.Timestamp('2026-06-14')
RECENT_DAYS   = 60   # fenêtre récente
PAST_DAYS     = 60   # fenêtre historique de comparaison

recent_start  = REFERENCE_DATE - timedelta(days=RECENT_DAYS)
past_start    = REFERENCE_DATE - timedelta(days=RECENT_DAYS + PAST_DAYS)

recent = decks[decks['date'] >= recent_start]
past   = decks[(decks['date'] >= past_start) & (decks['date'] < recent_start)]

def period_score(df, label):
    total = len(df)
    if total == 0:
        return pd.DataFrame(columns=['archetype', f'share_{label}', f'placement_score_{label}'])
    g = (df.groupby('archetype')
         .agg(count=('id','count'), avg_pl=('placement','mean'))
         .reset_index())
    g[f'share_{label}'] = g['count'] / total
    g[f'placement_score_{label}'] = 1.0 / g['avg_pl']
    g[f'meta_score_{label}'] = np.sqrt(
        g[f'share_{label}'] * g[f'placement_score_{label}'] /
        g[f'placement_score_{label}'].max()
    )
    return g[['archetype', f'share_{label}', f'meta_score_{label}']]

recent_scores = period_score(recent, 'recent')
past_scores   = period_score(past,   'past')

trend = recent_scores.merge(past_scores, on='archetype', how='outer').fillna(0)
trend['trend_ratio'] = np.where(
    trend['meta_score_past'] > 0,
    trend['meta_score_recent'] / trend['meta_score_past'],
    np.where(trend['meta_score_recent'] > 0, 3.0, 1.0)  # nouveau archetype → fort signal
)
trend['trend_label'] = pd.cut(
    trend['trend_ratio'],
    bins=[0, 0.5, 0.8, 1.25, 2.0, 999],
    labels=['⬇️ chute forte', '↘️ déclin', '➡️ stable', '↗️ montée', '⬆️ émergence']
)
trend = trend.sort_values('meta_score_recent', ascending=False)

print(f'Archetypes analysés : {len(trend)}')
print()
print('=== TOP 20 par score récent ===')
print(trend.head(20)[['archetype','share_recent','meta_score_recent','trend_ratio','trend_label']]
      .to_string(index=False, float_format='{:.3f}'.format))
print()
print('=== ÉMERGENCES (ratio > 2x) ===')
emerging = trend[trend['trend_ratio'] > 2.0].sort_values('trend_ratio', ascending=False)
print(emerging.head(10)[['archetype','share_recent','meta_score_recent','trend_ratio']]
      .to_string(index=False, float_format='{:.3f}'.format))
print()
print('=== CHUTES (ratio < 0.5) ===')
falling = trend[trend['trend_ratio'] < 0.5].sort_values('trend_ratio')
print(falling.head(10)[['archetype','share_recent','meta_score_recent','trend_ratio']]
      .to_string(index=False, float_format='{:.3f}'.format))

## 3. Time series — évolution des top archetypes

In [ ]:
# Identifier les archetypes les plus importants sur toute la période
top_archetypes = (monthly.groupby('archetype')['deck_count']
                  .sum()
                  .sort_values(ascending=False)
                  .head(10)
                  .index.tolist())

# Pivot pour time series
ts = (monthly[monthly['archetype'].isin(top_archetypes)]
      .pivot_table(index='month', columns='archetype', values='meta_score', fill_value=0))

print('Time series meta_score (top 10 archetypes) :')
print(ts.round(3).to_string())

## 4. Sauvegarder en base

In [ ]:
con2 = sqlite3.connect('../data/yugioh.db')

# Table meta_scores : score mensuel par archetype
con2.execute("DROP TABLE IF EXISTS meta_scores")
con2.execute("""
    CREATE TABLE meta_scores (
        month            TEXT,
        archetype        TEXT,
        deck_count       INTEGER,
        total_decks      INTEGER,
        share            REAL,
        avg_placement    REAL,
        meta_score       REAL,
        PRIMARY KEY (month, archetype)
    )
""")
monthly['month'] = monthly['month'].astype(str)
monthly[['month','archetype','deck_count','total_decks','share','avg_placement','meta_score']].to_sql(
    'meta_scores', con2, if_exists='append', index=False)

# Table archetype_trend : trend récent par archetype
con2.execute("DROP TABLE IF EXISTS archetype_trend")
con2.execute("""
    CREATE TABLE archetype_trend (
        archetype        TEXT PRIMARY KEY,
        share_recent     REAL,
        meta_score_recent REAL,
        meta_score_past  REAL,
        trend_ratio      REAL,
        trend_label      TEXT
    )
""")
trend['trend_label'] = trend['trend_label'].astype(str)
trend[['archetype','share_recent','meta_score_recent','meta_score_past','trend_ratio','trend_label']].to_sql(
    'archetype_trend', con2, if_exists='append', index=False)

con2.commit()
con2.close()

print(f'✓ meta_scores   : {len(monthly):,} lignes (mois × archetype)')
print(f'✓ archetype_trend : {len(trend):,} archetypes')